In [ ]:
# ============================================================
# FEATURE IMPORTANCE — RANDOM FOREST CLASSIFICATION
# ============================================================

# ----------------------------
# 1. IMPORT LIBRARIES
# ----------------------------
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score, make_scorer


# ----------------------------
# 2. CREATE OUTPUT FOLDER
# ----------------------------
os.makedirs("figures", exist_ok=True)


# ----------------------------
# 3. LOAD DATA
# ----------------------------
file_path = "example_dataset.xlsx"
df = pd.read_excel(file_path)

print("Dataset shape:", df.shape)


# ----------------------------
# 4. DEFINE INPUTS AND TARGET
# ----------------------------
x_cols = [
    "PERM_permeability",
    "PERM_perm_factor_of_failed_well",
    "PERM_dist_to_well_in_grid",
    "PERM_caprock_perm",
    "porosity",
    "aquifer_porosity",
    "aquifer_permeability"
]

target_col = "Leakage risk regime"

X = df[x_cols].values
y = df[target_col].values


# ============================================================
# PART A — GLOBAL FEATURE IMPORTANCE
# ============================================================

# ----------------------------
# 5A. TRAIN / TEST SPLIT
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ----------------------------
# 6A. TRAIN MULTICLASS RF
# ----------------------------
rf = RandomForestClassifier(
    n_estimators=350,
    min_samples_split=4,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)


# ----------------------------
# 7A. GLOBAL FEATURE IMPORTANCE
# ----------------------------
importances = rf.feature_importances_
sorted_idx = np.argsort(importances)
sorted_cols = np.array(x_cols)[sorted_idx]

plt.figure(figsize=(8, 5))
plt.barh(sorted_cols, importances[sorted_idx], color="#4A90E2")
plt.title("Random Forest — Global Feature Importance")
plt.xlabel("Importance Score")
plt.tight_layout()

plt.savefig(
    "figures/Figure_RF_GlobalFeatureImportance.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close()

print("\n=== Feature Importance (Global, sorted) ===")
for name, val in zip(sorted_cols, importances[sorted_idx]):
    print(f"{name}: {val:.5f}")


# ============================================================
# PART B — PER-CLASS PERMUTATION IMPORTANCE
# ============================================================

def per_class_feature_importance(class_name):
    print(f"\n==== Permutation Importance for class: {class_name} ====")

    # ----------------------------
    # 1. CREATE BINARY TARGET
    # ----------------------------
    y_binary = (df[target_col] == class_name).astype(int).values
    X_local = df[x_cols].values

    # ----------------------------
    # 2. TRAIN / TEST SPLIT
    # ----------------------------
    X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
        X_local,
        y_binary,
        test_size=0.25,
        random_state=42,
        stratify=y_binary
    )

    # ----------------------------
    # 3. TRAIN BINARY RF
    # ----------------------------
    rf_bin = RandomForestClassifier(
        n_estimators=350,
        min_samples_split=4,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )

    rf_bin.fit(X_train_c, y_train_c)

    # ----------------------------
    # 4. PERMUTATION IMPORTANCE
    # ----------------------------
    scorer = make_scorer(f1_score)

    result = permutation_importance(
        rf_bin,
        X_test_c,
        y_test_c,
        n_repeats=25,
        random_state=42,
        scoring=scorer,
        n_jobs=-1
    )

    perm_importances = result.importances_mean
    sorted_idx = np.argsort(perm_importances)
    sorted_features = np.array(x_cols)[sorted_idx]

    # ----------------------------
    # 5. PLOT
    # ----------------------------
    plt.figure(figsize=(7, 5))
    plt.barh(
        sorted_features,
        perm_importances[sorted_idx],
        color="#4A8FE7",
        alpha=0.85
    )
    plt.title(f"Feature Importance — {class_name}")
    plt.xlabel("Permutation Importance (F1-based)")
    plt.grid(alpha=0.3)
    plt.tight_layout()

    safe_name = class_name.replace(" ", "_")
    plt.savefig(
        f"figures/Figure_RF_PermutationImportance_{safe_name}.png",
        dpi=600,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    # ----------------------------
    # 6. PRINT VALUES
    # ----------------------------
    print(f"\nFeature Importance for {class_name}:")
    for name, val in zip(sorted_features, perm_importances[sorted_idx]):
        print(f"{name}: {val:.5f}")


# ----------------------------
# 8. RUN FOR EACH CLASS
# ----------------------------
per_class_feature_importance("low_risk")
per_class_feature_importance("high_risk")
per_class_feature_importance("extreme")